<a href="https://colab.research.google.com/github/hamed7pix/Data-Structures-and-Algorithms/blob/main/Queue_FIFO_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚶‍♂️🚶‍♀️🚶 Queue (FIFO) — Built From Scratch in Python

This notebook is part of a series that builds the fundamental data structures of an undergraduate **Data Structures** course from scratch in Python. Every structure follows the same pattern:

1. **Idea** — what problem the structure solves, explained with a picture or analogy.
2. **Implementation** — a complete, readable Python class (no library shortcuts for the structure itself).
3. **Operation tour** — a runnable example for each common operation:
   * 🔎 Access / peek / indexing
   * 🔍 Search / lookup / find
   * ➕ Insert (enqueue)
   * ➖ Delete (dequeue)
   * 🔁 Traversal, update, and specialized operations
4. **Complexity table** — time complexity first, then space.

We implement the queue **twice**:

| Part | Implementation | Backing storage |
|---|---|---|
| A | `ArrayQueue` | a Python `list` used as a fixed-size **circular array** (with resizing) |
| B | `LinkedQueue` | a **singly linked list** with `head` (front) and `tail` (rear) pointers |

> 🚫 We will **not** use `collections.deque` or `queue.Queue` — those *are* queues, and the point is to build one ourselves.

### Contents
0. Idea: what is a queue?
1. Warm-up: classes & objects in Python (quick refresher)
2. Why not just `list.append` + `list.pop(0)`?
3. Part A — `ArrayQueue` (circular array)
4. Part B — `LinkedQueue` (linked list)
5. Using the classes: creating and working with objects
6. Applications: simulation, hot potato, BFS
7. Testing both implementations against each other
8. Complexity tables
9. Exercises

---
## 0. 💡 Idea — What is a Queue?

A **queue** is a collection where items leave **in the same order they arrived**:

> **FIFO — First In, First Out.**

Think of the line at a bakery 🥖. New customers join at the **back (rear)**; the cashier serves whoever is at the **front**. Nobody cuts in line, and nobody leaves from the middle.

```
            dequeue ◀──                                ◀── enqueue
                     ┌───────┬───────┬───────┬───────┐
   leaves here  ◀──  │  Ana  │  Ben  │  Cy   │  Dee  │  ◀──  joins here
                     └───────┴───────┴───────┴───────┘
                       FRONT                    REAR
                    (oldest)                 (newest)
```

### The Queue ADT (Abstract Data Type)

| Operation | Meaning |
|---|---|
| `enqueue(x)` | add `x` at the **rear** |
| `dequeue()` | remove and return the item at the **front** |
| `peek()` / `front()` | look at the front item **without** removing it |
| `is_empty()` | `True` if there are no items |
| `len(q)` | number of items |

Real-world uses: printer jobs, CPU/task scheduling, network packet buffers, customer-service systems, keyboard buffers, and the **Breadth-First Search** algorithm on graphs.

**Queue vs Stack:** a stack is **LIFO** (last in, first out — like a pile of plates). A queue is **FIFO** (first in, first out — like a line of people).

---
## 1. 🧱 Warm-up: Classes and Objects in Python

Both queues will be written as **classes**, so here is a 2-minute refresher on the vocabulary.

* A **class** is a *blueprint* (e.g. "what every bank account looks like").
* An **object** (or **instance**) is one concrete thing built from that blueprint (e.g. "Ana's account").
* `__init__` is the **constructor**: it runs automatically when you create an object and sets up its **attributes** (its data).
* **Methods** are functions defined inside the class; the first parameter `self` is the object the method was called on.
* **Dunder** ("double underscore") methods like `__len__`, `__repr__`, `__iter__`, `__contains__` let our objects work with Python built-ins such as `len(obj)`, `print(obj)`, `for x in obj`, and `x in obj`.
* A leading underscore (`self._data`) is a convention meaning *"internal — please don't touch from outside."*

---
## 2. 🤔 Why not just `list.append` + `list.pop(0)`?

The "obvious" queue uses a Python list: append at the end, pop from index 0.
It *works*, but `pop(0)` must **shift every remaining element one slot left** — that is **O(n)** per dequeue.

```
pop(0):   [A, B, C, D, E]   →   remove A, then shift B, C, D, E left
           ↑  ←  ←  ←  ←
```

Let's write it (for comparison only) and measure.

In [2]:
class NaiveListQueue:
    """Correct but slow: dequeue is O(n) because pop(0) shifts all elements."""
    def __init__(self):
        self._data = []
    def enqueue(self, item):
        self._data.append(item)          # O(1) amortized
    def dequeue(self):
        if not self._data:
            raise IndexError("dequeue from empty queue")
        return self._data.pop(0)         # O(n)  <-- the problem
    def __len__(self):
        return len(self._data)

In [3]:
import time

def time_fill_and_drain(queue_class, n):
    q = queue_class()
    start = time.perf_counter()
    for i in range(n):
        q.enqueue(i)
    while len(q):
        q.dequeue()
    return time.perf_counter() - start

for n in (10_000, 40_000, 160_000):
    t = time_fill_and_drain(NaiveListQueue, n)
    print(f"NaiveListQueue  n={n:>7,}: {t*1000:8.1f} ms")
# Watch the growth: 4x more items -> much more than 4x the time (quadratic total work).
# We'll re-run this benchmark with our two real queues later.

NaiveListQueue  n= 10,000:     13.2 ms
NaiveListQueue  n= 40,000:    157.8 ms
NaiveListQueue  n=160,000:   3559.0 ms


**Goal:** make *both* `enqueue` and `dequeue` **O(1)**. Two classic solutions:

* **A.** Never shift — instead let the *front index move*, and wrap around the end of the array → **circular array**.
* **B.** Use a **linked list** where removing the first node is just moving a pointer.

---
## 3. Part A — `ArrayQueue`: Queue with a Circular Array

### 3.1 💡 Idea

We allocate a list of fixed **capacity** (e.g. 8 slots, filled with `None`) and keep two numbers:

* `_front` — index of the front item
* `_size` — how many items are stored

The rear position is **computed**, not stored:

$$\text{rear index} = (\text{front} + \text{size}) \bmod \text{capacity}$$

The `% capacity` makes the indices **wrap around**, so the array behaves like a ring 🔄:

```
capacity = 8, front = 5, size = 5

 index:   0     1     2     3     4     5     6     7
        ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
        │  D  │  E  │  ·  │  ·  │  ·  │  A  │  B  │  C  │
        └─────┴─────┴─────┴─────┴─────┴─────┴─────┴─────┘
                       ▲                 ▲
                next free slot         front
           (5 + 5) % 8 = 2

 Logical order (front → rear):  A, B, C, D, E
```

* **enqueue** → write at `(front + size) % cap`, then `size += 1`. **No shifting.**
* **dequeue** → read at `front`, then `front = (front + 1) % cap`, `size -= 1`. **No shifting.**
* **Full?** (`size == capacity`) → **resize**: allocate a list twice as big and copy items in logical order to indices `0, 1, 2, …`. Rare, so enqueue is **O(1) amortized**.
* **Mostly empty?** (`size ≤ capacity/4`) → shrink to half to avoid wasting memory.

The *i*-th item from the front lives at physical index `(front + i) % capacity`.

### 3.2 🛠️ Implementation

In [ ]:
class ArrayQueue:
    """FIFO queue implemented with a circular array (a Python list of fixed capacity).

    Front of the queue is at index self._front.
    The i-th element from the front is at index (self._front + i) % capacity.
    """

    DEFAULT_CAPACITY = 8

    # ------------------------------------------------------------ construction
    def __init__(self, capacity=DEFAULT_CAPACITY, items=None):
        """Create an empty queue; optionally enqueue every element of `items`."""
        if capacity < 1:
            raise ValueError("capacity must be at least 1")
        self._data = [None] * capacity   # the raw array (fixed size)
        self._front = 0                  # index of the front element
        self._size = 0                   # number of stored elements
        if items is not None:
            for x in items:
                self.enqueue(x)

    # ------------------------------------------------------------ helpers
    def _capacity(self):
        return len(self._data)

    def _physical(self, i):
        """Translate logical position i (0 = front) into an index of self._data."""
        return (self._front + i) % self._capacity()

    def _check_position(self, i):
        if not 0 <= i < self._size:
            raise IndexError(f"position {i} out of range for queue of size {self._size}")

    def _resize(self, new_capacity):
        """Copy the elements into a new array, re-aligned so the front is at index 0."""
        old = self._data
        old_cap = len(old)
        self._data = [None] * new_capacity
        for i in range(self._size):
            self._data[i] = old[(self._front + i) % old_cap]
        self._front = 0

    # ------------------------------------------------------------ size / status
    def __len__(self):
        return self._size

    def is_empty(self):
        return self._size == 0

    # ------------------------------------------------------------ 🔎 access
    def peek(self):
        """Return (don't remove) the front element."""
        if self.is_empty():
            raise IndexError("peek from empty queue")
        return self._data[self._front]

    def peek_rear(self):
        """Return (don't remove) the most recently enqueued element."""
        if self.is_empty():
            raise IndexError("peek_rear from empty queue")
        return self._data[self._physical(self._size - 1)]

    def get(self, i):
        """Return the element at logical position i (0 = front). O(1) thanks to the array."""
        self._check_position(i)
        return self._data[self._physical(i)]

    # ------------------------------------------------------------ 🔍 search
    def search(self, item):
        """Return the position of `item` counted from the front (0-based), or -1."""
        for i in range(self._size):
            if self._data[self._physical(i)] == item:
                return i
        return -1

    def __contains__(self, item):        # enables:  item in q
        return self.search(item) != -1

    # ------------------------------------------------------------ ➕ insert
    def enqueue(self, item):
        """Add `item` at the rear."""
        if self._size == self._capacity():          # full -> double
            self._resize(2 * self._capacity())
        rear = self._physical(self._size)           # first free slot after the last item
        self._data[rear] = item
        self._size += 1

    # ------------------------------------------------------------ ➖ delete
    def dequeue(self):
        """Remove and return the front element."""
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        item = self._data[self._front]
        self._data[self._front] = None              # help the garbage collector
        self._front = (self._front + 1) % self._capacity()
        self._size -= 1
        # shrink when only a quarter full (but never below the default capacity)
        if (self._capacity() > self.DEFAULT_CAPACITY
                and self._size <= self._capacity() // 4):
            self._resize(self._capacity() // 2)
        return item

    def clear(self):
        """Remove every element."""
        self._data = [None] * self.DEFAULT_CAPACITY
        self._front = 0
        self._size = 0

    # ------------------------------------------------------------ 🔁 traversal / update / special
    def __iter__(self):
        """Traverse from front to rear (does NOT remove anything)."""
        for i in range(self._size):
            yield self._data[self._physical(i)]

    def update(self, i, value):
        """Replace the element at logical position i. (Not a classic queue op — handy for teaching.)"""
        self._check_position(i)
        self._data[self._physical(i)] = value

    def rotate(self, k=1):
        """Move the front element to the rear, k times (round-robin)."""
        if self.is_empty():
            return
        for _ in range(k % self._size):
            self.enqueue(self.dequeue())

    # ------------------------------------------------------------ display
    def __repr__(self):
        return "ArrayQueue(front -> [" + ", ".join(repr(x) for x in self) + "] <- rear)"

    def debug_view(self):
        """Show the raw array, with F marking the front slot and R the last element."""
        cap = self._capacity()
        rear = self._physical(self._size - 1) if self._size else None
        cells = []
        for idx, val in enumerate(self._data):
            tag = ""
            if self._size and idx == self._front: tag += "F"
            if idx == rear: tag += "R"
            cells.append(f"{idx}:{'·' if val is None else val}{'('+tag+')' if tag else ''}")
        print(f"cap={cap} front={self._front} size={self._size} | " + "  ".join(cells))

### 3.3 🧭 Operation Tour — `ArrayQueue`

We'll use a small capacity (4) so we can *see* wrap-around and resizing happen.

#### ➕ Insert — `enqueue`

In [ ]:
q = ArrayQueue(capacity=4)        # create an object of class ArrayQueue
print(q, "| empty?", q.is_empty())

for name in ["Ana", "Ben", "Cy"]:
    q.enqueue(name)
    q.debug_view()

print(q, "| len =", len(q))

ArrayQueue(front -> [] <- rear) | empty? True
cap=4 front=0 size=1 | 0:Ana(FR)  1:·  2:·  3:·
cap=4 front=0 size=2 | 0:Ana(F)  1:Ben(R)  2:·  3:·
cap=4 front=0 size=3 | 0:Ana(F)  1:Ben  2:Cy(R)  3:·
ArrayQueue(front -> ['Ana', 'Ben', 'Cy'] <- rear) | len = 3


#### 🔎 Access — `peek`, `peek_rear`, `get(i)`

In [ ]:
print("front item :", q.peek())        # Ana  (not removed)
print("rear item  :", q.peek_rear())   # Cy
print("position 1 :", q.get(1))        # Ben
print("still      :", q)               # nothing was removed

front item : Ana
rear item  : Cy
position 1 : Ben
still      : ArrayQueue(front -> ['Ana', 'Ben', 'Cy'] <- rear)


#### 🔍 Search — `search`, `in`

In [ ]:
print("search('Cy')  ->", q.search("Cy"))      # 2 (two people ahead of Cy)
print("search('Zed') ->", q.search("Zed"))     # -1, not found
print("'Ben' in q    ->", "Ben" in q)          # uses __contains__

search('Cy')  -> 2
search('Zed') -> -1
'Ben' in q    -> True


#### ➖ Delete — `dequeue`
Watch `front` move to the right instead of items shifting left.

In [ ]:
served = q.dequeue()
print("served:", served)
q.debug_view()
print(q)

served: Ana
cap=4 front=1 size=2 | 0:·  1:Ben(F)  2:Cy(R)  3:·
ArrayQueue(front -> ['Ben', 'Cy'] <- rear)


#### 🔄 Wrap-around and resizing
Now enqueue more items. The rear goes past the last index and **wraps** to index 0; when the array is full it **doubles**.

In [ ]:
q.enqueue("Dee")      # goes to index 0? let's see...
q.debug_view()
q.enqueue("Eve")      # array has 4 slots and 4 items now -> full
q.debug_view()
q.enqueue("Fin")      # full -> resize to 8, items re-aligned starting at index 0
q.debug_view()
print(q)

cap=4 front=1 size=3 | 0:·  1:Ben(F)  2:Cy  3:Dee(R)
cap=4 front=1 size=4 | 0:Eve(R)  1:Ben(F)  2:Cy  3:Dee
cap=8 front=0 size=5 | 0:Ben(F)  1:Cy  2:Dee  3:Eve  4:Fin(R)  5:·  6:·  7:·
ArrayQueue(front -> ['Ben', 'Cy', 'Dee', 'Eve', 'Fin'] <- rear)


#### 🔁 Traversal — `for x in q` (non-destructive)

In [ ]:
for position, person in enumerate(q):
    print(f"#{position}: {person}")
print("queue unchanged:", q)

#0: Ben
#1: Cy
#2: Dee
#3: Eve
#4: Fin
queue unchanged: ArrayQueue(front -> ['Ben', 'Cy', 'Dee', 'Eve', 'Fin'] <- rear)


#### 🔁 Update — `update(i, value)`

In [ ]:
q.update(0, "BEN")          # fix the spelling of the front person
print(q)

ArrayQueue(front -> ['BEN', 'Cy', 'Dee', 'Eve', 'Fin'] <- rear)


#### 🔁 Specialized — `rotate(k)` (round-robin: front goes to the back)

In [ ]:
print("before :", q)
q.rotate(2)
print("rotate2:", q)

before : ArrayQueue(front -> ['BEN', 'Cy', 'Dee', 'Eve', 'Fin'] <- rear)
rotate2: ArrayQueue(front -> ['Dee', 'Eve', 'Fin', 'BEN', 'Cy'] <- rear)


#### ⚠️ Errors on an empty queue + `clear`

In [ ]:
q.clear()
print(q, "| empty?", q.is_empty())

try:
    q.dequeue()
except IndexError as err:
    print("IndexError:", err)

try:
    q.peek()
except IndexError as err:
    print("IndexError:", err)

ArrayQueue(front -> [] <- rear) | empty? True
IndexError: dequeue from empty queue
IndexError: peek from empty queue


---
## 4. Part B — `LinkedQueue`: Queue with a Linked List

### 4.1 💡 Idea

A **singly linked list** is a chain of **nodes**. Each node stores a value and a reference (`next`) to the following node.
We keep **two** references:

* `head` → the **front** node (where we dequeue)
* `tail` → the **rear** node (where we enqueue)

```
   head (front)                                tail (rear)
      │                                           │
      ▼                                           ▼
  ┌─────┬───┐     ┌─────┬───┐     ┌─────┬───┐     ┌─────┬──────┐
  │ Ana │ ●─┼───▶ │ Ben │ ●─┼───▶ │ Cy  │ ●─┼───▶ │ Dee │ None │
  └─────┴───┘     └─────┴───┘     └─────┴───┘     └─────┴──────┘
```

* **enqueue(x)** → create a node, link `tail.next` to it, move `tail` to it. **O(1)**
* **dequeue()** → remember `head.data`, move `head` to `head.next`. **O(1)**
* Special case: if the list becomes empty, `tail` must also become `None`.

Why must the *front* be the **head**? Removing the head of a singly linked list is O(1), but removing the tail would need the node *before* it, which takes O(n) to find. So we **dequeue at the head** and **enqueue at the tail**.

No capacity, no resizing — each item gets its own node. The cost: an extra reference per item, and no O(1) indexing.

### 4.2 🛠️ Implementation

In [ ]:
class Node:
    """One link of the chain: a value plus a reference to the next node."""
    __slots__ = ("data", "next")       # saves memory; optional

    def __init__(self, data, next=None):
        self.data = data
        self.next = next

    def __repr__(self):
        return f"Node({self.data!r})"


class LinkedQueue:
    """FIFO queue implemented with a singly linked list.
    head = front (dequeue here), tail = rear (enqueue here).
    """

    # ------------------------------------------------------------ construction
    def __init__(self, items=None):
        self._head = None     # front node
        self._tail = None     # rear node
        self._size = 0
        if items is not None:
            for x in items:
                self.enqueue(x)

    # ------------------------------------------------------------ helpers
    def _node_at(self, i):
        """Walk i steps from the head. O(i)."""
        if not 0 <= i < self._size:
            raise IndexError(f"position {i} out of range for queue of size {self._size}")
        node = self._head
        for _ in range(i):
            node = node.next
        return node

    # ------------------------------------------------------------ size / status
    def __len__(self):
        return self._size

    def is_empty(self):
        return self._size == 0

    # ------------------------------------------------------------ 🔎 access
    def peek(self):
        if self.is_empty():
            raise IndexError("peek from empty queue")
        return self._head.data

    def peek_rear(self):
        if self.is_empty():
            raise IndexError("peek_rear from empty queue")
        return self._tail.data

    def get(self, i):
        """Element at logical position i (0 = front). O(i) — must walk the chain."""
        return self._node_at(i).data

    # ------------------------------------------------------------ 🔍 search
    def search(self, item):
        """Position of `item` from the front (0-based), or -1."""
        node, i = self._head, 0
        while node is not None:
            if node.data == item:
                return i
            node, i = node.next, i + 1
        return -1

    def __contains__(self, item):
        return self.search(item) != -1

    # ------------------------------------------------------------ ➕ insert
    def enqueue(self, item):
        new_node = Node(item)
        if self.is_empty():
            self._head = new_node          # first node is both front and rear
        else:
            self._tail.next = new_node     # link old rear -> new node
        self._tail = new_node              # new node is the rear
        self._size += 1

    # ------------------------------------------------------------ ➖ delete
    def dequeue(self):
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        old_head = self._head
        self._head = old_head.next         # unlink the front node
        self._size -= 1
        if self._head is None:             # queue became empty
            self._tail = None              # don't leave tail pointing at a removed node!
        old_head.next = None               # tidy up
        return old_head.data

    def clear(self):
        self._head = self._tail = None
        self._size = 0

    # ------------------------------------------------------------ 🔁 traversal / update / special
    def __iter__(self):
        node = self._head
        while node is not None:
            yield node.data
            node = node.next

    def update(self, i, value):
        self._node_at(i).data = value

    def rotate(self, k=1):
        """Move the front element to the rear, k times.
        Done by relinking nodes — no new Node objects are created."""
        if self._size < 2:
            return
        for _ in range(k % self._size):
            node = self._head
            self._head = node.next
            node.next = None
            self._tail.next = node
            self._tail = node

    # ------------------------------------------------------------ display
    def __repr__(self):
        return "LinkedQueue(front -> [" + ", ".join(repr(x) for x in self) + "] <- rear)"

    def debug_view(self):
        """Draw the chain of nodes."""
        if self.is_empty():
            print("head=None  tail=None  (empty)")
            return
        chain = " -> ".join(f"[{x}]" for x in self) + " -> None"
        print(f"head={self._head.data!r} tail={self._tail.data!r} size={self._size} | {chain}")

### 4.3 🧭 Operation Tour — `LinkedQueue`

#### ➕ Insert — `enqueue`

In [ ]:
lq = LinkedQueue()
lq.debug_view()
for job in ["print.pdf", "photo.png", "report.docx"]:
    lq.enqueue(job)
    lq.debug_view()

head=None  tail=None  (empty)
head='print.pdf' tail='print.pdf' size=1 | [print.pdf] -> None
head='print.pdf' tail='photo.png' size=2 | [print.pdf] -> [photo.png] -> None
head='print.pdf' tail='report.docx' size=3 | [print.pdf] -> [photo.png] -> [report.docx] -> None


#### 🔎 Access — `peek`, `peek_rear`, `get(i)`

In [ ]:
print("front :", lq.peek())
print("rear  :", lq.peek_rear())
print("get(1):", lq.get(1), "  <- had to walk 1 link from the head")

front : print.pdf
rear  : report.docx
get(1): photo.png   <- had to walk 1 link from the head


#### 🔍 Search — `search`, `in`

In [ ]:
print(lq.search("report.docx"))   # 2
print(lq.search("missing.txt"))   # -1
print("photo.png" in lq)

2
-1
True


#### ➖ Delete — `dequeue` (including the empty-queue special case)

In [ ]:
while not lq.is_empty():
    print("printing:", lq.dequeue())
    lq.debug_view()      # notice tail becomes None when the last node leaves

try:
    lq.dequeue()
except IndexError as err:
    print("IndexError:", err)

printing: print.pdf
head='photo.png' tail='report.docx' size=2 | [photo.png] -> [report.docx] -> None
printing: photo.png
head='report.docx' tail='report.docx' size=1 | [report.docx] -> None
printing: report.docx
head=None  tail=None  (empty)
IndexError: dequeue from empty queue


#### 🔁 Traversal, update, rotate

In [ ]:
lq = LinkedQueue(items=[10, 20, 30, 40])   # constructor can take initial items
print(lq)

total = 0
for value in lq:                 # traversal
    total += value
print("sum by traversal:", total)

lq.update(2, 33)                 # update position 2
print("after update :", lq)

lq.rotate(1)                     # 10 moves to the rear by relinking
print("after rotate :", lq)
lq.debug_view()

LinkedQueue(front -> [10, 20, 30, 40] <- rear)
sum by traversal: 100
after update : LinkedQueue(front -> [10, 20, 33, 40] <- rear)
after rotate : LinkedQueue(front -> [20, 33, 40, 10] <- rear)
head=20 tail=10 size=4 | [20] -> [33] -> [40] -> [10] -> None


---
## 5. 🧑‍🏫 Using the Classes: Creating and Working with Objects

This section focuses on **object-oriented usage**: creating several objects, keeping their states independent, putting *our own* objects inside a queue, and writing code that works with *either* queue class.

### 5.1 Several independent objects

In [ ]:
morning = ArrayQueue()            # object 1
evening = ArrayQueue()            # object 2  (same class, different object)
vip     = LinkedQueue(["Mia"])    # object 3  (different class)

morning.enqueue("Ana")
morning.enqueue("Ben")
evening.enqueue("Zoe")

print("morning:", morning)
print("evening:", evening)
print("vip    :", vip)
print("Same object?", morning is evening)    # False — each has its own _data, _front, _size
print("morning attributes:", vars(morning))  # peek inside the object's attributes

morning: ArrayQueue(front -> ['Ana', 'Ben'] <- rear)
evening: ArrayQueue(front -> ['Zoe'] <- rear)
vip    : LinkedQueue(front -> ['Mia'] <- rear)
Same object? False
morning attributes: {'_data': ['Ana', 'Ben', None, None, None, None, None, None], '_front': 0, '_size': 2}


### 5.2 A queue can hold *any* objects — including objects of our own class

Let's define a small `Customer` class and a `BankTeller` class that **has a queue inside it** (this is called *composition*: an object made of other objects).

In [ ]:
class Customer:
    """A customer waiting in line."""
    next_ticket = 1                       # class attribute: shared by all Customer objects

    def __init__(self, name, task):
        self.name = name                  # instance attributes: unique per object
        self.task = task
        self.ticket = Customer.next_ticket
        Customer.next_ticket += 1

    def __repr__(self):
        return f"Customer(#{self.ticket} {self.name}: {self.task})"


class BankTeller:
    """A teller that serves customers in FIFO order. Works with ANY queue class."""

    def __init__(self, name, queue_class=ArrayQueue):
        self.name = name
        self.line = queue_class()          # composition: the teller *has a* queue
        self.served = []

    def arrive(self, customer):
        self.line.enqueue(customer)
        print(f"  {customer.name} joins {self.name}'s line (position {len(self.line) - 1})")

    def serve_next(self):
        if self.line.is_empty():
            print(f"  {self.name}: nobody waiting ☕")
            return None
        c = self.line.dequeue()
        self.served.append(c)
        print(f"  {self.name} serves {c}")
        return c

    def __repr__(self):
        return f"BankTeller({self.name}, waiting={len(self.line)}, served={len(self.served)})"


# Create objects
t1 = BankTeller("Window 1")                           # uses ArrayQueue (default)
t2 = BankTeller("Window 2", queue_class=LinkedQueue)  # same code, linked-list queue

alice = Customer("Alice", "deposit")
bob   = Customer("Bob", "open account")
cara  = Customer("Cara", "withdraw")
dan   = Customer("Dan", "loan question")

print("Arrivals:")
t1.arrive(alice); t1.arrive(bob); t2.arrive(cara); t1.arrive(dan)

print("\nWho is next at Window 1?", t1.line.peek())
print("Is Bob waiting at Window 1?", bob in t1.line)
print("Bob's position:", t1.line.search(bob))

print("\nServing:")
t1.serve_next(); t2.serve_next(); t2.serve_next(); t1.serve_next()

print("\n", t1, "\n", t2, sep="")

Arrivals:
  Alice joins Window 1's line (position 0)
  Bob joins Window 1's line (position 1)
  Cara joins Window 2's line (position 0)
  Dan joins Window 1's line (position 2)

Who is next at Window 1? Customer(#1 Alice: deposit)
Is Bob waiting at Window 1? True
Bob's position: 1

Serving:
  Window 1 serves Customer(#1 Alice: deposit)
  Window 2 serves Customer(#3 Cara: withdraw)
  Window 2: nobody waiting ☕
  Window 1 serves Customer(#2 Bob: open account)

BankTeller(Window 1, waiting=1, served=2)
BankTeller(Window 2, waiting=0, served=1)


### 5.3 Same interface, different implementation (duck typing / polymorphism)

Because `ArrayQueue` and `LinkedQueue` offer the **same methods**, a function can accept either one without caring which.

In [ ]:
def drain(queue):
    """Dequeue everything and return it as a list. Works for any object with is_empty/dequeue."""
    out = []
    while not queue.is_empty():
        out.append(queue.dequeue())
    return out

for QueueClass in (ArrayQueue, LinkedQueue):
    q = QueueClass(items="FIFO!")          # a string is iterable: 'F','I','F','O','!'
    print(f"{QueueClass.__name__:12} {q}  ->  drained: {drain(q)}  -> now {q}")

ArrayQueue   ArrayQueue(front -> ['F', 'I', 'F', 'O', '!'] <- rear)  ->  drained: ['F', 'I', 'F', 'O', '!']  -> now ArrayQueue(front -> [] <- rear)
LinkedQueue  LinkedQueue(front -> ['F', 'I', 'F', 'O', '!'] <- rear)  ->  drained: ['F', 'I', 'F', 'O', '!']  -> now LinkedQueue(front -> [] <- rear)


### 5.4 Queues of tuples (e.g. events with timestamps)

In [ ]:
events = LinkedQueue()
events.enqueue(("09:00", "door opened"))
events.enqueue(("09:02", "alarm disarmed"))
events.enqueue(("09:05", "coffee machine on"))

while not events.is_empty():
    time_stamp, what = events.dequeue()      # tuple unpacking
    print(f"[{time_stamp}] {what}")

[09:00] door opened
[09:02] alarm disarmed
[09:05] coffee machine on


---
## 6. 🚀 Applications

### 6.1 Printer simulation (FIFO scheduling)
Jobs arrive over time; the printer prints one page per second. How long does each job wait?

In [ ]:
class PrintJob:
    def __init__(self, owner, pages, arrival):
        self.owner, self.pages, self.arrival = owner, pages, arrival
    def __repr__(self):
        return f"{self.owner}({self.pages}p)"

arrivals = {0: PrintJob("ana", 3, 0), 1: PrintJob("ben", 1, 1),
            2: PrintJob("cy", 2, 2),  6: PrintJob("dee", 1, 6)}

waiting = ArrayQueue()
current, remaining = None, 0

for second in range(10):
    if second in arrivals:
        waiting.enqueue(arrivals[second])
    if current is None and not waiting.is_empty():
        current = waiting.dequeue()
        remaining = current.pages
        print(f"t={second}: start {current} (waited {second - current.arrival}s)")
    if current is not None:
        remaining -= 1
        if remaining == 0:
            print(f"t={second}: finished {current}")
            current = None
    print(f"         waiting: {list(waiting)}")

t=0: start ana(3p) (waited 0s)
         waiting: []
         waiting: [ben(1p)]
t=2: finished ana(3p)
         waiting: [ben(1p), cy(2p)]
t=3: start ben(1p) (waited 2s)
t=3: finished ben(1p)
         waiting: [cy(2p)]
t=4: start cy(2p) (waited 2s)
         waiting: []
t=5: finished cy(2p)
         waiting: []
t=6: start dee(1p) (waited 0s)
t=6: finished dee(1p)
         waiting: []
         waiting: []
         waiting: []
         waiting: []


### 6.2 Hot Potato (Josephus problem) — uses `rotate`
Children stand in a circle passing a potato. After `k` passes, whoever holds it is out. Last one standing wins.
A queue models the circle: passing = moving the front to the rear.

In [ ]:
def hot_potato(names, k, queue_class=LinkedQueue):
    circle = queue_class(items=names)
    while len(circle) > 1:
        circle.rotate(k)                       # pass the potato k times
        print(f"  out: {circle.dequeue():6} remaining: {list(circle)}")
    return circle.peek()

print("Winner:", hot_potato(["Bill", "David", "Susan", "Jane", "Kent", "Brad"], 7))

  out: David  remaining: ['Susan', 'Jane', 'Kent', 'Brad', 'Bill']
  out: Kent   remaining: ['Brad', 'Bill', 'Susan', 'Jane']
  out: Jane   remaining: ['Brad', 'Bill', 'Susan']
  out: Bill   remaining: ['Susan', 'Brad']
  out: Brad   remaining: ['Susan']
Winner: Susan


### 6.3 Breadth-First Search (BFS) — the queue's most famous job
BFS explores a graph **level by level**. The queue guarantees we visit all nodes at distance 1 before any at distance 2, and so on — which gives the **shortest path in an unweighted graph**.

In [ ]:
campus = {
    "Gate":    ["Library", "Cafe"],
    "Library": ["Gate", "Lab"],
    "Cafe":    ["Gate", "Gym", "Lab"],
    "Lab":     ["Library", "Cafe", "Dorm"],
    "Gym":     ["Cafe"],
    "Dorm":    ["Lab"],
}

def bfs_shortest_path(graph, start, goal):
    frontier = ArrayQueue()
    frontier.enqueue(start)
    parent = {start: None}                    # also acts as the 'visited' set
    while not frontier.is_empty():
        here = frontier.dequeue()             # oldest discovered node first (FIFO!)
        if here == goal:
            break
        for nxt in graph[here]:
            if nxt not in parent:
                parent[nxt] = here
                frontier.enqueue(nxt)
    if goal not in parent:
        return None
    path, node = [], goal                      # walk parents back to the start
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

print(" -> ".join(bfs_shortest_path(campus, "Gate", "Dorm")))

Gate -> Library -> Lab -> Dorm


---
## 7. ✅ Testing Both Implementations Against Each Other

We perform thousands of random operations on both queues **and** on a trusted reference (a plain list used carefully), and check they always agree. Then we re-run the benchmark from Section 2.

In [ ]:
import random

def check_same(ref, *queues):
    for q in queues:
        assert list(q) == ref, (q, ref)
        assert len(q) == len(ref)
        assert q.is_empty() == (len(ref) == 0)
        if ref:
            assert q.peek() == ref[0] and q.peek_rear() == ref[-1]

random.seed(42)
ref, aq, lq = [], ArrayQueue(capacity=2), LinkedQueue()

for step in range(5_000):
    op = random.random()
    if op < 0.55:                                   # enqueue
        x = random.randint(0, 99)
        ref.append(x); aq.enqueue(x); lq.enqueue(x)
    elif op < 0.90 and ref:                         # dequeue
        expected = ref.pop(0)
        assert aq.dequeue() == expected == lq.dequeue()
    elif ref:                                       # search / get / update / rotate
        x = random.randint(0, 99)
        expected = ref.index(x) if x in ref else -1
        assert aq.search(x) == expected == lq.search(x)
        i = random.randrange(len(ref))
        assert aq.get(i) == ref[i] == lq.get(i)
        aq.update(i, -1); lq.update(i, -1); ref[i] = -1
        k = random.randint(0, 5)
        aq.rotate(k); lq.rotate(k)
        for _ in range(k % len(ref)):
            ref.append(ref.pop(0))
    check_same(ref, aq, lq)

print("All 5,000 random operations agree ✅")

All 5,000 random operations agree ✅


In [ ]:
for n in (10_000, 40_000, 160_000):
    row = [f"n={n:>7,}"]
    for cls in (NaiveListQueue, ArrayQueue, LinkedQueue):
        row.append(f"{cls.__name__}: {time_fill_and_drain(cls, n)*1000:7.1f} ms")
    print(" | ".join(row))
# ArrayQueue and LinkedQueue grow ~linearly (O(1) per op); NaiveListQueue grows ~quadratically.

n= 10,000 | NaiveListQueue:     7.3 ms | ArrayQueue:     5.8 ms | LinkedQueue:     3.4 ms
n= 40,000 | NaiveListQueue:   125.7 ms | ArrayQueue:    24.0 ms | LinkedQueue:    15.5 ms


n=160,000 | NaiveListQueue:  2010.9 ms | ArrayQueue:   133.3 ms | LinkedQueue:   111.8 ms


---
## 8. 📊 Complexity Tables

*n* = number of items in the queue. "Amortized" means: averaged over a long sequence of operations (an occasional O(n) resize is paid for by many cheap O(1) operations).

### 8.1 ⏱️ Time complexity

| Operation | `ArrayQueue` (circular array) | `LinkedQueue` (linked list) | `NaiveListQueue` (`pop(0)`) |
|---|---|---|---|
| ➕ `enqueue(x)` | **O(1)** amortized (O(n) when resizing) | **O(1)** | O(1) amortized |
| ➖ `dequeue()` | **O(1)** amortized (O(n) when shrinking) | **O(1)** | **O(n)** ❌ |
| 🔎 `peek()` / `peek_rear()` | O(1) | O(1) | O(1) |
| 🔎 `get(i)` (i-th from front) | **O(1)** | O(i) → O(n) worst | O(1) |
| 🔍 `search(x)` / `x in q` | O(n) | O(n) | O(n) |
| 🔁 traversal (`for x in q`) | O(n) | O(n) | O(n) |
| 🔁 `update(i, v)` | O(1) | O(i) → O(n) worst | O(1) |
| 🔁 `rotate(k)` | O(k) amortized | O(k) (relinks only) | O(k·n) |
| `len(q)`, `is_empty()` | O(1) | O(1) | O(1) |
| `clear()` | O(1)* | O(1)* | O(1)* |

\* Python's garbage collector frees the old items afterwards.

### 8.2 💾 Space complexity

| | `ArrayQueue` | `LinkedQueue` |
|---|---|---|
| Total storage | **O(n)** — capacity is kept between *n* and about 4*n* (doubling + shrink-at-¼ policy) | **O(n)** — exactly *n* nodes |
| Overhead per item | unused empty slots (up to ~3 per item right after a shrink-trigger point) | one extra `next` reference per node + node object overhead |
| Extra space per operation | O(1); O(n) temporarily during a resize | O(1) |
| Memory layout | contiguous block → cache-friendly, fast in practice | scattered nodes → more pointer chasing |

### 8.3 Which one should I use?

* **`ArrayQueue`** — great default: O(1) indexing, compact, cache-friendly. Cost: occasional resize spikes.
* **`LinkedQueue`** — every enqueue/dequeue is *truly* O(1) (no spikes), never wastes empty slots. Cost: more memory per item, slow indexing.
* In real Python code you'd use **`collections.deque`**, which is internally a linked list of fixed-size array blocks — a hybrid of both ideas!

---
## 9. 📝 Exercises

1. **Bounded queue.** Write `BoundedArrayQueue(capacity)` that **never resizes**: `enqueue` on a full queue raises `OverflowError`. Add an `is_full()` method.
2. **Queue from two stacks.** Implement a queue using only two Python lists used as stacks (`append`/`pop()` only). Show that dequeue is O(1) *amortized*.
3. **`__eq__`.** Add `__eq__` to both classes so that `ArrayQueue([1,2]) == LinkedQueue([1,2])` is `True`.
4. **Reverse.** Write `reverse(q)` that reverses a queue in place using only queue operations plus one stack (a list).
5. **Deque.** Extend `LinkedQueue` to a **double-ended queue**: add `add_front` and `remove_rear`. Which one is hard with a *singly* linked list, and why? Fix it with a *doubly* linked list.
6. **Trace by hand.** Starting from `ArrayQueue(capacity=4)`, perform `enqueue(1..3), dequeue(), dequeue(), enqueue(4..6)`. Draw the raw array, `front`, and `size` after every step, then check with `debug_view()`.
7. **Circular buffer without `_size`.** Store `front` and `rear` instead of `size`. How do you tell *full* from *empty*? (Hint: leave one slot unused.)

In [ ]:
# ✍️ Your solutions here